# E-Commerce Data Analysis

Este notebook contiene el análisis principal del proyecto utilizando MongoDB, Aggregation Pipelines, Pandas y visualizaciones en Python.

El objetivo es validar la información cargada en MongoDB, calcular los principales indicadores de negocio, analizar tendencias y generar hallazgos útiles a partir del dataset.

## 1. Configuración y conexión a MongoDB

In [1]:
from pymongo import MongoClient
import pandas as pd
import matplotlib.pyplot as plt
import requests

In [2]:
MONGODB_URI = "mongodb://localhost:27017"
DATABASE_NAME = "ecommerce_analytics"

client = MongoClient(MONGODB_URI)
db = client[DATABASE_NAME]

client.admin.command("ping")

print("Conexión a MongoDB correcta.")
print("Colecciones disponibles:")
print(db.list_collection_names())

Conexión a MongoDB correcta.
Colecciones disponibles:
['times', 'sales', 'stores', 'items', 'payments', 'customers']


## 2. Validación de colecciones

In [3]:
collections = [
    "customers",
    "items",
    "stores",
    "times",
    "payments",
    "sales"
]

for collection_name in collections:
    count = db[collection_name].count_documents({})
    print(f"{collection_name}: {count:,}")

customers: 9,191
items: 264
stores: 726
times: 99,999
payments: 39
sales: 1,000,000


## 3. Indicadores principales

In [4]:
pipeline = [
    {
        "$group": {
            "_id": None,
            "total_sales": {"$sum": "$total_price"},
            "transactions": {"$sum": 1},
            "unique_customers": {"$addToSet": "$customer_key"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "total_sales": 1,
            "transactions": 1,
            "unique_customers": {"$size": "$unique_customers"},
            "average_ticket": {
                "$divide": ["$total_sales", "$transactions"]
            }
        }
    }
]

kpis = list(db.sales.aggregate(pipeline))[0]

print(f"Venta total: {kpis['total_sales']:,.2f}")
print(f"Número de transacciones: {kpis['transactions']:,}")
print(f"Clientes únicos: {kpis['unique_customers']:,}")
print(f"Ticket promedio: {kpis['average_ticket']:,.2f}")

Venta total: 105,401,435.75
Número de transacciones: 1,000,000
Clientes únicos: 9,191
Ticket promedio: 105.40


## 4. Evolución de ventas

In [5]:
pipeline = [
    {
        "$lookup": {
            "from": "times",
            "localField": "time_key",
            "foreignField": "time_key",
            "as": "time_info"
        }
    },
    {
        "$unwind": "$time_info"
    },
    {
        "$group": {
            "_id": {
                "year": "$time_info.year",
                "month": "$time_info.month"
            },
            "total_sales": {
                "$sum": "$total_price"
            }
        }
    },
    {
        "$sort": {
            "_id.year": 1,
            "_id.month": 1
        }
    }
]

sales_evolution = list(db.sales.aggregate(pipeline))

In [ ]:
df_sales_evolution = pd.DataFrame([
    {
        "year": row["_id"]["year"],
        "month": row["_id"]["month"],
        "total_sales": row["total_sales"]
    }
    for row in sales_evolution
])

df_sales_evolution.head(12)

In [ ]:
df_sales_evolution["date"] = pd.to_datetime(
    dict(
        year=df_sales_evolution["year"],
        month=df_sales_evolution["month"],
        day=1
    )
)

df_sales_evolution.head()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    df_sales_evolution["date"],
    df_sales_evolution["total_sales"]
)

plt.title("Evolución mensual de ventas")
plt.xlabel("Fecha")
plt.ylabel("Venta total")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

## 5. Comparación de tiendas

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": "$store_key",
            "total_sales": {
                "$sum": "$total_price"
            },
            "transactions": {
                "$sum": 1
            }
        }
    },
    {
        "$lookup": {
            "from": "stores",
            "localField": "_id",
            "foreignField": "store_key",
            "as": "store_info"
        }
    },
    {
        "$unwind": "$store_info"
    },
    {
        "$project": {
            "_id": 0,
            "store_key": "$_id",
            "division": "$store_info.division",
            "district": "$store_info.district",
            "upazila": "$store_info.upazila",
            "total_sales": 1,
            "transactions": 1
        }
    },
    {
        "$sort": {
            "total_sales": -1
        }
    }
]

sales_by_store = list(db.sales.aggregate(pipeline))

df_sales_by_store = pd.DataFrame(sales_by_store)

df_sales_by_store.head(10)

In [ ]:
top_stores = df_sales_by_store.head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    top_stores["upazila"],
    top_stores["total_sales"]
)

plt.title("Top 10 tiendas por venta total")
plt.xlabel("Venta total")
plt.ylabel("Tienda")

plt.gca().invert_yaxis()
plt.tight_layout()

plt.show()

## 6. Comparación de productos

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": "$item_key",
            "total_sales": {
                "$sum": "$total_price"
            },
            "quantity_sold": {
                "$sum": "$quantity"
            },
            "transactions": {
                "$sum": 1
            }
        }
    },
    {
        "$lookup": {
            "from": "items",
            "localField": "_id",
            "foreignField": "item_key",
            "as": "item_info"
        }
    },
    {
        "$unwind": "$item_info"
    },
    {
        "$project": {
            "_id": 0,
            "item_key": "$_id",
            "item_name": "$item_info.item_name",
            "supplier": "$item_info.supplier",
            "total_sales": 1,
            "quantity_sold": 1,
            "transactions": 1
        }
    },
    {
        "$sort": {
            "total_sales": -1
        }
    }
]

sales_by_product = list(db.sales.aggregate(pipeline))

df_sales_by_product = pd.DataFrame(sales_by_product)

df_sales_by_product.head(10)

In [ ]:
top_products = df_sales_by_product.head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    top_products["item_name"],
    top_products["total_sales"]
)

plt.title("Top 10 productos por venta total")
plt.xlabel("Venta total")
plt.ylabel("Producto")

plt.gca().invert_yaxis()
plt.tight_layout()

plt.show()

## 7. Análisis por método de pago

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": "$payment_key",
            "total_sales": {
                "$sum": "$total_price"
            },
            "transactions": {
                "$sum": 1
            }
        }
    },
    {
        "$lookup": {
            "from": "payments",
            "localField": "_id",
            "foreignField": "payment_key",
            "as": "payment_info"
        }
    },
    {
        "$unwind": "$payment_info"
    },
    {
        "$project": {
            "_id": 0,
            "payment_key": "$_id",
            "transaction_type": "$payment_info.transaction_type",
            "bank_name": "$payment_info.bank_name",
            "total_sales": 1,
            "transactions": 1
        }
    },
    {
        "$sort": {
            "total_sales": -1
        }
    }
]

sales_by_payment = list(db.sales.aggregate(pipeline))

df_sales_by_payment = pd.DataFrame(sales_by_payment)

df_sales_by_payment.head(10)

In [ ]:
payment_summary = (
    df_sales_by_payment
    .groupby("transaction_type", as_index=False)
    .agg({
        "total_sales": "sum",
        "transactions": "sum"
    })
    .sort_values("total_sales", ascending=False)
)

payment_summary

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    payment_summary["transaction_type"],
    payment_summary["total_sales"]
)

plt.title("Ventas por método de pago")
plt.xlabel("Método de pago")
plt.ylabel("Venta total")

plt.tight_layout()
plt.show()

## 8. Validaciones y posibles anomalías

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": None,
            "min_sale": {"$min": "$total_price"},
            "max_sale": {"$max": "$total_price"},
            "avg_sale": {"$avg": "$total_price"}
        }
    }
]

sale_stats = list(db.sales.aggregate(pipeline))[0]

print(f"Venta mínima: {sale_stats['min_sale']:,.2f}")
print(f"Venta máxima: {sale_stats['max_sale']:,.2f}")
print(f"Venta promedio: {sale_stats['avg_sale']:,.2f}")

## 9. Consumo de API pública

In [ ]:
url = "https://dummyjson.com/products"

response = requests.get(url, timeout=10)
response.raise_for_status()

data = response.json()

df_api = pd.DataFrame(data["products"])[
    ["id", "title", "category", "price", "rating"]
]

df_api.head(10)

## 10. Salida ejecutiva
Parte III - Visualización y análisis de negocio

Esta sección consolida los principales resultados del análisis y permite observar la situación general de ventas, su evolución y el comportamiento de productos, tiendas y métodos de pago.

In [ ]:
print("RESUMEN GENERAL")
print("-" * 45)

print(f"Venta total: {kpis['total_sales']:,.2f}")
print(f"Número de transacciones: {kpis['transactions']:,}")
print(f"Clientes únicos: {kpis['unique_customers']:,}")
print(f"Ticket promedio: {kpis['average_ticket']:,.2f}")

### Segmentación por año

Se utiliza el año como criterio de segmentación para observar el comportamiento de las ventas dentro de un periodo específico.

In [ ]:
selected_year = 2019

df_year = df_sales_evolution[
    df_sales_evolution["year"] == selected_year
].copy()

df_year

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    df_year["date"],
    df_year["total_sales"]
)

plt.title(f"Evolución mensual de ventas - {selected_year}")
plt.xlabel("Fecha")
plt.ylabel("Venta total")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Principales resultados

- `Red Bull 12oz` fue el producto con mayor venta total.
- Las tiendas líderes presentan niveles de venta relativamente cercanos entre sí.
- El método de pago predominante es `card`, muy por encima de `mobile` y `cash`.
- La evolución mensual muestra variaciones a lo largo del periodo, aunque sin cambios extremos sostenidos durante la mayor parte del análisis.

### Conclusión ejecutiva

El análisis muestra un volumen total de ventas de 105,401,435.75 distribuido en 1,000,000 de transacciones, con 9,191 clientes únicos y un ticket promedio de 105.40.

La segmentación por año permite analizar periodos específicos sin necesidad de trabajar nuevamente con todas las transacciones.

A nivel comercial, destacan la alta participación de los pagos con tarjeta, el liderazgo de `Red Bull 12oz` por venta total y un desempeño relativamente equilibrado entre las tiendas con mayores ventas.

Estos resultados pueden servir como punto de partida para análisis posteriores relacionados con rentabilidad, inventario, comportamiento de clientes y desempeño por ubicación.